# 06 - Model Evaluation, Error Analysis, XAI (Module-First)

Notebook nay chi orchestration va goi truc tiep cac module evaluation:
- `compute_ops_metrics`
- `compute_pr_metrics_for_rare_classes`
- `build_evaluation_report_markdown`

Yeu cau du lieu that:
- File predictions (parquet/csv) chua `y_true`, `y_pred`
- Khuyen nghi co them `score_class_4`, `score_class_5` de tinh PR metrics lop hiem
- Co the nap them benchmark summary json de bao cao latency

In [ ]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

from src.rl.evaluation import (
    compute_ops_metrics,
    compute_pr_metrics_for_rare_classes,
    build_evaluation_report_markdown,
)

In [ ]:
# ==== Input config ====
ROOT = Path('/workspace/ai-core')

PREDICTIONS_PATH = Path(
    os.getenv(
        'EVAL_PREDICTIONS_PATH',
        str(ROOT / 'artifacts' / 'rl' / 'evaluation' / 'predictions.parquet')
    )
)

# Optional benchmark summary (from scripts.experimental.benchmark_datamart)
BENCHMARK_SUMMARY_PATH = Path(
    os.getenv(
        'EVAL_BENCHMARK_SUMMARY_PATH',
        str(ROOT / 'artifacts' / 'rl' / 'benchmarks' / 'rl_benchmark_pilot_summary.json')
    )
)

REPORT_OUT = Path(
    os.getenv(
        'EVAL_REPORT_OUT',
        str(ROOT / 'notebooks' / '06_evaluation_report.md')
    )
)

assert PREDICTIONS_PATH.exists(), (
    f'Missing predictions file: {PREDICTIONS_PATH}. '
    'Please export real predictions with columns y_true, y_pred.'
)

if PREDICTIONS_PATH.suffix.lower() == '.parquet':
    eval_df = pd.read_parquet(PREDICTIONS_PATH)
elif PREDICTIONS_PATH.suffix.lower() == '.csv':
    eval_df = pd.read_csv(PREDICTIONS_PATH)
else:
    raise ValueError(f'Unsupported prediction file format: {PREDICTIONS_PATH.suffix}')

required_cols = ['y_true', 'y_pred']
missing = [c for c in required_cols if c not in eval_df.columns]
assert not missing, f'Missing required columns in predictions file: {missing}'

y_true = pd.to_numeric(eval_df['y_true'], errors='coerce').fillna(0).astype(int).to_numpy()
y_pred = pd.to_numeric(eval_df['y_pred'], errors='coerce').fillna(0).astype(int).to_numpy()

print('Predictions file:', PREDICTIONS_PATH)
print('Rows:', len(eval_df))
print('Columns:', list(eval_df.columns))

In [ ]:
# Ops metrics + normalized confusion matrix
ops_metrics = compute_ops_metrics(y_true=y_true, y_pred=y_pred)

cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4, 5], normalize='true')

print('Operational metrics:')
for k, v in ops_metrics.items():
    print(f'- {k}: {v}')

print('\nNormalized confusion matrix:')
print(np.round(cm_norm, 4))

In [ ]:
# Rare-class PR metrics (requires score columns)
score_cols = {
    4: 'score_class_4',
    5: 'score_class_5',
}

available_score_map = {}
for cls, col in score_cols.items():
    if col in eval_df.columns:
        available_score_map[cls] = pd.to_numeric(eval_df[col], errors='coerce').fillna(0.0).to_numpy()

if available_score_map:
    pr_metrics = compute_pr_metrics_for_rare_classes(
        y_true=y_true,
        y_score_by_class=available_score_map,
        rare_classes=tuple(sorted(available_score_map.keys())),
    )
    print('PR metrics (rare classes):')
    print(pr_metrics)
else:
    pr_metrics = {'num_samples': int(len(y_true)), 'classes': {}}
    print('No score columns found (score_class_4/score_class_5). Skip PR metrics.')

In [ ]:
# Optional latency summary + build markdown report
latency_payload = {}
if BENCHMARK_SUMMARY_PATH.exists():
    with open(BENCHMARK_SUMMARY_PATH, 'r', encoding='utf-8') as f:
        latency_payload = json.load(f)

extra_sections = {
    'Data Source': {
        'predictions_path': str(PREDICTIONS_PATH),
        'rows': int(len(eval_df)),
    },
    'Latency Benchmark Summary': latency_payload if latency_payload else 'Not available',
}

report_md = build_evaluation_report_markdown(
    run_name=os.getenv('EVAL_RUN_NAME', 'notebook06_eval'),
    ops_metrics=ops_metrics,
    pr_metrics=pr_metrics,
    extra_sections=extra_sections,
)

REPORT_OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT_OUT.write_text(report_md, encoding='utf-8')

print('Saved report:', REPORT_OUT)
print('\nReport preview:\n')
print('\n'.join(report_md.splitlines()[:30]))

In [ ]:
# Optional export of confusion matrix for downstream plotting
eval_artifacts_dir = ROOT / 'notebooks' / '06_eval_artifacts'
eval_artifacts_dir.mkdir(parents=True, exist_ok=True)

cm_out = eval_artifacts_dir / 'confusion_matrix_normalized.csv'
pd.DataFrame(cm_norm, columns=[f'pred_{i}' for i in range(6)]).to_csv(cm_out, index=False)

ops_out = eval_artifacts_dir / 'ops_metrics.json'
with open(ops_out, 'w', encoding='utf-8') as f:
    json.dump(ops_metrics, f, indent=2)

pr_out = eval_artifacts_dir / 'pr_metrics.json'
with open(pr_out, 'w', encoding='utf-8') as f:
    json.dump(pr_metrics, f, indent=2)

print('Saved:', cm_out)
print('Saved:', ops_out)
print('Saved:', pr_out)